# Teste isolado — Instituto Trata Brasil (Blog)

Fonte candidata: **Instituto Trata Brasil**, setor Saneamento. Notebook
**descartável** (Fase 1) — sem dispatcher, sem `atualizar_status_fonte`,
sem gravar nada. Só valida:

1. Scraping da listagem do blog (título, categoria, data, resumo, link)
2. Extração do texto completo de um post

## Confirmado antes de assumir

WordPress padrão confirmado (`robots.txt` simples, sem bloqueios
relevantes). Tema usa **Elementor** com loop de posts em containers
`div.e-loop-item` (não é o loop clássico `article.post` nem um page
builder de terceiros como GT3/jeg) — dentro de cada item:

- Categoria: `.elementor-post-info__terms-list-item a`
- Título/link: `h1.elementor-heading-title a`
- Resumo: `.elementor-widget-theme-post-excerpt .elementor-widget-container`
- Data: `[itemprop="datePublished"] time` (formato `DD/MM/YYYY`)

Paginação confirmada em `/blog/2/`, `/blog/3/`, ... até `/blog/30/`
(10 posts por página, ~300 no total) — conteúdo genuinamente diferente
entre páginas (testado).

Texto completo do post em `.elementor-widget-theme-post-content` (não
`.entry-content`, que não existe nesse tema — outro detalhe do Elementor).

Conteúdo é mais analítico/institucional (estudos, rankings, dados de
investimento) do que notícia factual pontual — sem filtro de relevância
aqui, como pedido; isso é responsabilidade da etapa de NLP.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://tratabrasil.org.br/blog/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
ITENS_POR_PAGINA_ESPERADO = 10

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`div.e-loop-item` por post -- título/link em `h1.elementor-heading-title a`,
categoria em `.elementor-post-info__terms-list-item a`, resumo em
`.elementor-widget-theme-post-excerpt .elementor-widget-container`, data em
`[itemprop="datePublished"] time` (`DD/MM/YYYY`). Paginação `/blog/N/`.
~300 posts no total (30 páginas) — limito a amostra aqui.

In [0]:
def listar_tratabrasil(max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/{pagina}/"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        itens_pagina = soup.select("div.e-loop-item")
        if not itens_pagina:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        for item in itens_pagina:
            tag_a = item.select_one("h1.elementor-heading-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            categoria = None
            tag_categoria = item.select_one(".elementor-post-info__terms-list-item")
            if tag_categoria:
                categoria = tag_categoria.get_text(strip=True)

            resumo = None
            tag_resumo = item.select_one(".elementor-widget-theme-post-excerpt .elementor-widget-container")
            if tag_resumo:
                resumo = tag_resumo.get_text(" ", strip=True)

            data_publicacao = None
            tag_data = item.select_one('[itemprop="datePublished"] time')
            if tag_data:
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "categoria": categoria,
                "resumo": resumo,
                "published_at": data_publicacao,
            })

        print(f"  página {pagina}: {len(itens_pagina)} itens.")
        if len(itens_pagina) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_tratabrasil()

print(f"\n{len(itens)} posts listados.\n")
print(f"{'DATA':<12} {'CATEGORIA':<20} TÍTULO")
print("-" * 110)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {(item['categoria'] or '?')[:20]:<20} {item['titulo'][:70]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")
print(f"Exemplo de resumo: {itens[0]['resumo']}")

## Teste 2 — abrir um post e extrair o texto completo

`.elementor-widget-theme-post-content` -- widget de conteúdo do Elementor,
diferente de `.entry-content` (não existe nesse tema).

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".elementor-widget-theme-post-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(" ", strip=True) if h1 else None


def extrair_post_tratabrasil(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    titulo = extrair_titulo_h1(html) or item["titulo"]

    return {
        "titulo": titulo,
        "url": item["url"],
        "categoria": item["categoria"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_post_tratabrasil(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} posts abertos com sucesso.")
curtos = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtos)}")

In [0]:
# Amostra completa do primeiro post — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"CATEGORIA   : {detalhe['categoria']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/categoria/resumo/
data/link de todos os itens, texto completo sai limpo via
`.elementor-widget-theme-post-content`. Amostra confirma o que o pedido
antecipava — conteúdo mais analítico/institucional (estudos, rankings,
dados de investimento) do que notícia factual pontual de agência
reguladora. Sem filtro de relevância aplicado, como pedido.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` — sem Selenium, sem parsing que dependa de JS. Precisa
de uma `listar_tratabrasil()` própria (paginação `/blog/N`, seletores do
tema Elementor `e-loop-item`/`elementor-post-info__terms-list-item`, data
em `DD/MM/YYYY`), mas o texto do detalhe reaproveita
`extrair_texto_generico()` com `.elementor-widget-theme-post-content`
acrescentado a `SELETORES_CONTEUDO` — sem extrator próprio.

Histórico médio (~300 posts, 30 páginas) -- `max_paginas` conservador
(recente, não backfill completo), mesmo critério de ANP/ABEGÁS/ABAR.